# Módulo 2 · Machine Learning — Forecast de demanda y clasificación de riesgo

- **2A Regresión:** RandomForest vs XGBoost con validación walk-forward (sin data leakage).
- **2B Clasificación:** riesgo de desabasto con manejo de desbalance (`scale_pos_weight`).
- **2C Interpretabilidad:** SHAP / feature importance.

## Datos: la serie temporal de demanda

52 semanas por punto, generadas con `seed=42` como `base × (1 + tendencia) × estacionalidad + ruido`. La estacionalidad es una onda anual con fase distinta por punto. Abajo, 3 puntos de ejemplo.

In [1]:
import sys; sys.path.append("..")
from src.utils import set_seeds, generate_demand_series
from src.ml_models import train_forecast_models, train_risk_classifier, interpretability, forecast_demand
import matplotlib.pyplot as plt
set_seeds()
series = generate_demand_series()
# Serie de 3 puntos de ejemplo: estacionalidad + ruido
fig, ax = plt.subplots(figsize=(10,4))
for pid in ["D1","D6","D17"]:
    s = series[series.point_id==pid]
    ax.plot(s.week, s.demand, label=pid)
ax.legend(); ax.set_xlabel("semana"); ax.set_ylabel("demanda"); plt.show()

/var/folders/hf/bs3qcwv97ls8w_9tvykhcqp80000gn/T/ipykernel_15442/2113780637.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); ax.set_xlabel("semana"); ax.set_ylabel("demanda"); plt.show()


## 2A · Pronóstico con validación walk-forward

Comparamos RandomForest vs XGBoost. La validación es **walk-forward** (ventana expansiva, 4 folds × 4 semanas): se entrena siempre con el pasado y se valida en el futuro inmediato, por lo que **no hay fuga de datos**. Métricas: MAE, RMSE y MAPE.

In [2]:
# 2A · Walk-forward: 4 folds de 4 semanas, ventana expansiva
fc = train_forecast_models()
print(f"Mejor modelo: {fc.best_name}")
fc.metrics

Mejor modelo: RandomForest


,MAE,RMSE,MAPE
model,,,
RandomForest,14.265,19.434,8.162
XGBoost,15.054,20.702,8.452


In [3]:
# Forecast recursivo a 4 semanas para un punto
forecast_demand(fc, "D17", weeks_ahead=4)

[89.4, 98.2, 96.2, 98.7]

## 2B · Clasificador de riesgo de desabasto

Split temporal: 40 semanas de entrenamiento / 12 de prueba. **Integración 2A→2B:** la feature `projected_demand` es la salida del modelo de forecast (entrenado solo con semanas < 40, sin fuga), no un valor aleatorio. Esto elimina la circularidad del diseño original — las métricas bajan un poco (de ~0.91 a ~0.87 de AUC) pero son **honestas**. El desbalance de clases se maneja con `scale_pos_weight`.

In [4]:
# 2B · Clasificador de riesgo (split temporal 40/12 semanas)
rk = train_risk_classifier()
print(rk.metrics)
print(rk.confusion)

{'ROC_AUC': 0.8667, 'F1': 0.7673, 'positive_rate_train': 0.258, 'scale_pos_weight': 2.88}
[[142  18]
 [ 19  61]]


## 2C · Interpretabilidad (SHAP)

Importancia global de cada feature = promedio del valor absoluto de su SHAP (TreeSHAP exacto). Indica qué variable empuja más la probabilidad de riesgo.

In [5]:
# 2C · Interpretabilidad
interp = interpretability(fc, rk)
print(interp["risk_method"])
interp["risk_importance"]

/Users/cagera/Documents/Claude/Projects/Ternium/logistics-challenge/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP (mean |value|, TreeSHAP)


current_stock       2.348906
projected_demand    1.294394
lead_time_days      0.511710
distance_km         0.415416
dtype: float32

**Conclusión ejecutiva:** la variable más determinante del riesgo de desabasto es `current_stock`, seguida de `projected_demand`. Lead time y distancia son amplificadores secundarios. Operativamente: priorizar políticas de stock de seguridad sobre renegociación de rutas.